# Tokamak Disruption Simulator — Colab Pipeline

**Purpose:** Generate synthetic tokamak disruption training data using
FreeGSNKE (MHD equilibrium) + TORAX (1D transport) coupled simulation.

**Output:** HDF5 dataset saved to Google Drive with:
- Normal shots (label=0): Te, ne, j, q profiles over time
- Disruptive shots (label=1): pre-disruption evolution + trigger crossing

## Steps
1. Install dependencies
2. Clone the repository
3. Verify TORAX works (JAX + AVX available in Colab)
4. Run a single test shot (verify pipeline)
5. Run batch dataset generation
6. Save to Google Drive

## Step 1: Install Dependencies

In [ ]:
# Install FreeGSNKE with the Newton-Krylov GS solver (freegs4e)
!pip install -q "freegsnke[freegs4e]"

# Install TORAX (includes JAX — works in Colab because Colab CPUs have AVX)
!pip install -q torax

# Other dependencies
!pip install -q h5py matplotlib numpy scipy

In [ ]:
# Verify all imports work
import freegsnke
import freegs4e
import jax
import torax
import numpy as np
import h5py

print(f"freegsnke : {freegsnke.__version__}")
print(f"freegs4e  : {freegs4e.__version__}")
print(f"jax       : {jax.__version__}")
print(f"torax     : {torax.__version__}")
print(f"numpy     : {np.__version__}")
print()

# Quick JAX smoke test
x = jax.numpy.array([1.0, 2.0, 3.0])
print(f"JAX OK — devices: {jax.devices()}")

## Step 2: Clone Repository & Set Up Paths

In [ ]:
import os

REPO_URL  = "https://github.com/BecerraMiguel/tokamak-disruption-simulator.git"
REPO_DIR  = "/content/tokamak-disruption-simulator"
DINA_URL  = "https://github.com/iterorganization/DINA-IMAS.git"
DINA_DIR  = "/content/DINA-IMAS"

# Clone the simulator repo
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Clone DINA-IMAS for the ITER machine config
if not os.path.exists(DINA_DIR):
    # Only need the machines/ directory — use sparse checkout for speed
    !git clone --no-checkout --depth=1 {DINA_URL} {DINA_DIR}
    !cd {DINA_DIR} && git sparse-checkout init --cone
    !cd {DINA_DIR} && git sparse-checkout set machines
    !cd {DINA_DIR} && git checkout

# Add repo src/ to Python path
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

print("Repository ready.")
print(f"ITER config: {DINA_DIR}/machines/iter/tokamak_config.dat")
assert os.path.exists(f"{DINA_DIR}/machines/iter/tokamak_config.dat"), \
    "ITER config not found — check DINA-IMAS clone"

## Step 3: Build the ITER Machine

In [ ]:
from predisruption.iter_machine import build_iter_machine, ITER_PARAMS

CONFIG_PATH = f"{DINA_DIR}/machines/iter/tokamak_config.dat"

tokamak, active_coils, domain = build_iter_machine(
    config_path=CONFIG_PATH,
    include_passives=False,   # Phase 1: no eddy currents
    verbose=True,
)
print(f"\nDomain: R=[{domain[0]}, {domain[1]}] m, Z=[{domain[2]}, {domain[3]}] m")
print(f"Active circuits: {list(active_coils.keys())}")

## Step 4: Compute Initial ITER Equilibrium

In [ ]:
from predisruption.equilibrium import EquilibriumSolver

eq_solver = EquilibriumSolver(
    tokamak=tokamak,
    nx=65, ny=65,
    Rmin=domain[0], Rmax=domain[1],
    Zmin=domain[2], Zmax=domain[3],
    verbose=True,
)

print("Solving initial 15 MA equilibrium...")
eq = eq_solver.solve_static(
    Ip=15.0e6,
    betap=0.5,
    xpoints=[(6.0, -3.8)],
)

signals = eq_solver.get_signals(eq)
print(f"\nInitial equilibrium:")
print(f"  Ip    = {signals['Ip']*1e-6:.2f} MA")
print(f"  q95   = {signals['q95']:.2f}")
print(f"  betaN = {signals['betaN']:.3f}")
print(f"  kappa = {signals['kappa']:.2f}")
print(f"  delta = {signals['delta']:.2f}")

In [ ]:
import matplotlib.pyplot as plt
from predisruption.shot_runner import TRIGGERS

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Poloidal flux map
R = eq.R
Z = eq.Z
psi = eq.psi()
axes[0].contourf(R, Z, psi.T, levels=20, cmap='RdBu_r')
R_sep, Z_sep = signals['R_sep'], signals['Z_sep']
if len(R_sep) > 0:
    axes[0].plot(R_sep, Z_sep, 'k-', lw=2, label='Separatrix')
axes[0].plot(signals['R_axis'], signals['Z_axis'], 'k+', ms=10, label='O-point')
axes[0].set_xlabel('R (m)')
axes[0].set_ylabel('Z (m)')
axes[0].set_title('Poloidal flux ψ(R,Z)')
axes[0].set_aspect('equal')
axes[0].legend()

# q profile
rho = signals['rho']
q   = signals['q_profile']
axes[1].plot(rho, q, 'b-', lw=2)
axes[1].axhline(y=1.0, ls='--', c='gray', label='q=1')
axes[1].axhline(y=2.0, ls='--', c='orange', label='q=2')
axes[1].axhline(y=TRIGGERS['q95'], ls='--', c='red', label=f"q95 limit={TRIGGERS['q95']}")
axes[1].set_xlabel('ρ = √(Φ/Φ_edge)')
axes[1].set_ylabel('q(ρ)')
axes[1].set_title('Safety factor profile')
axes[1].legend()
axes[1].set_ylim(0, 8)

plt.tight_layout()
plt.savefig(f"{REPO_DIR}/results/iter_initial_eq.png", dpi=120)
plt.show()
print("Saved to results/iter_initial_eq.png")

## Step 5: Run a Single Test Shot (Normal)

In [ ]:
from predisruption.transport import TransportSolver
from predisruption.shot_runner import ShotRunner, reference_15MA_scenario, TRIGGERS

# Transport solver — will use TORAX since JAX is available in Colab
tr_solver = TransportSolver(backend="auto", n_rho=51)

# Shot runner
runner = ShotRunner(
    eq_solver=eq_solver,
    tr_solver=tr_solver,
    verbose=True,
)

# Shorter test shot (20 s flat-top) to verify the pipeline
scenario = reference_15MA_scenario(t_end=30.0, dt=1.0)
result = runner.run_shot(scenario, shot_id=0)

print(f"\nTest shot complete:")
print(f"  label          = {result['label']}")
print(f"  time steps     = {len(result['time'])}")
print(f"  Te(0, t_end)   = {result['T_e'][0, -1]:.1f} keV")
print(f"  ne(0, t_end)   = {result['n_e'][0, -1]:.2e} m^-3")
print(f"  q95(t_end)     = {result['q95'][-1]:.2f}")
print(f"  betaN(t_end)   = {result['betaN'][-1]:.3f}")
print(f"  f_GW(t_end)    = {result['f_GW'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
t = result['time']

axes[0,0].plot(t, result['Ip']*1e-6)
axes[0,0].set(ylabel='Ip (MA)', title='Plasma current')

axes[0,1].plot(t, result['q95'])
axes[0,1].axhline(2.2, ls='--', c='red', label='limit')
axes[0,1].set(ylabel='q95', title='Safety factor at 95%')
axes[0,1].legend()

axes[0,2].plot(t, result['betaN'])
axes[0,2].axhline(3.2, ls='--', c='red', label='limit')
axes[0,2].set(ylabel='βN', title='Normalised beta')
axes[0,2].legend()

axes[1,0].plot(t, result['f_GW'])
axes[1,0].axhline(0.95, ls='--', c='red', label='limit')
axes[1,0].set(ylabel='f_GW', title='Greenwald fraction')
axes[1,0].legend()

# Te profile at last time step
rho = result['rho']
axes[1,1].plot(rho, result['T_e'][:, -1])
axes[1,1].set(xlabel='ρ', ylabel='Te (keV)', title=f'Te profile at t={t[-1]:.0f}s')

# ne profile at last time step
axes[1,2].plot(rho, result['n_e'][:, -1]*1e-20)
axes[1,2].set(xlabel='ρ', ylabel='ne (10²⁰ m⁻³)', title=f'ne profile at t={t[-1]:.0f}s')

for ax in axes.flat:
    ax.set_xlabel(ax.get_xlabel() or 't (s)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Test shot — ITER 15 MA normal', fontsize=14)
plt.tight_layout()
plt.savefig(f"{REPO_DIR}/results/test_shot.png", dpi=120)
plt.show()

## Step 6: Batch Dataset Generation

Generate N_normal normal shots + N_disruptive disruptive shots.
Vary: Ip (5–15 MA), P_heat (10–40 MW), ne_target (0.5–1.2 × n_GW).

In [ ]:
# ---- Dataset generation parameters ----
N_NORMAL      = 50    # normal shots per run
N_DISRUPTIVE  = 50    # disruptive shots per run
T_END_NORMAL  = 90.0  # s
T_END_DISRUPT = 75.0  # s
DT_COUPLE     = 1.0   # coupling time step (s)
RNG_SEED      = 42

rng = np.random.default_rng(RNG_SEED)
print(f"Will generate {N_NORMAL} normal + {N_DISRUPTIVE} disruptive shots")

In [ ]:
import h5py
import os
from predisruption.shot_runner import disruptive_scenario, reference_15MA_scenario

OUTPUT_H5 = f"{REPO_DIR}/data/dataset_v1.h5"
os.makedirs(os.path.dirname(OUTPUT_H5), exist_ok=True)

def vary_normal_scenario(rng, t_end, dt):
    """Sample a random normal scenario from the ITER operating space."""
    Ip_flat = rng.uniform(8e6, 15e6)       # 8–15 MA
    P_heat  = rng.uniform(20e6, 40e6)      # 20–40 MW
    ne_frac = rng.uniform(0.5, 0.85)       # 50–85% Greenwald
    a = ITER_PARAMS['a_minor']
    n_GW = (Ip_flat*1e-6) / (np.pi * a**2) * 1e20   # m^-3
    ne_target = ne_frac * n_GW
    cfg = reference_15MA_scenario(t_end=t_end, dt=dt)
    cfg.Ip_val  = np.array([3e6, Ip_flat, Ip_flat, 3e6])
    cfg.P_heat_val = np.array([5e6, P_heat, P_heat])
    cfg.ne_val  = np.array([3e19, ne_target, ne_target])
    return cfg

def vary_disruptive_scenario(rng, t_end, dt):
    """Sample a random disruptive scenario."""
    ptype = rng.choice(["density", "beta", "q95"])
    amp   = rng.uniform(0.2, 0.5)
    t_perturb = rng.uniform(30.0, min(50.0, t_end - 10.0))
    return disruptive_scenario(
        perturbation_type=ptype,
        perturbation_start=t_perturb,
        perturbation_amp=amp,
        t_end=t_end, dt=dt,
    )

def write_shot_to_hdf5(f, result, shot_id):
    """Write one shot result to an open HDF5 file."""
    grp = f.create_group(f"shots/shot_{shot_id:05d}")
    grp.attrs["label"]           = result["label"]
    grp.attrs["disruption_time"] = result["disruption_time"]
    grp.attrs["trigger"]         = result["trigger"]
    grp.attrs["shot_id"]         = shot_id

    grp.create_dataset("time", data=result["time"].astype(np.float32))
    grp.create_dataset("rho",  data=result["rho"].astype(np.float32))

    sig = grp.create_group("signals")
    for key in ["Ip", "betaN", "q95", "f_GW", "W_thermal"]:
        sig.create_dataset(key, data=result[key].astype(np.float32))
    for key in ["T_e", "n_e", "j_tor", "q_profile"]:
        sig.create_dataset(key, data=result[key].astype(np.float32))

# Run generation
with h5py.File(OUTPUT_H5, "w") as hf:
    # Metadata
    meta = hf.create_group("metadata")
    meta.attrs["n_normal"]      = N_NORMAL
    meta.attrs["n_disruptive"]  = N_DISRUPTIVE
    meta.attrs["dt_couple"]     = DT_COUPLE
    meta.attrs["torax_backend"] = str(tr_solver.backend)
    meta.attrs["rng_seed"]      = RNG_SEED

    shot_count = 0

    # --- Normal shots ---
    print(f"Generating {N_NORMAL} normal shots...")
    for i in range(N_NORMAL):
        try:
            scenario = vary_normal_scenario(rng, T_END_NORMAL, DT_COUPLE)
            result   = runner.run_shot(scenario, shot_id=shot_count)
            write_shot_to_hdf5(hf, result, shot_count)
            shot_count += 1
            print(f"  Normal {i+1}/{N_NORMAL} done (shot_{shot_count-1:05d})")
        except Exception as e:
            print(f"  Normal shot {i} FAILED: {e}")

    # --- Disruptive shots ---
    print(f"\nGenerating {N_DISRUPTIVE} disruptive shots...")
    for i in range(N_DISRUPTIVE):
        try:
            scenario = vary_disruptive_scenario(rng, T_END_DISRUPT, DT_COUPLE)
            result   = runner.run_shot(scenario, shot_id=shot_count)
            write_shot_to_hdf5(hf, result, shot_count)
            shot_count += 1
            print(f"  Disruptive {i+1}/{N_DISRUPTIVE} done  "
                  f"trigger={result['trigger']}  "
                  f"t_disrupt={result['disruption_time']:.1f}s")
        except Exception as e:
            print(f"  Disruptive shot {i} FAILED: {e}")

print(f"\nDataset written: {OUTPUT_H5}")
print(f"Total shots: {shot_count}")

## Step 7: Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DEST = "/content/drive/MyDrive/tokamak_disruption_data/"
os.makedirs(DRIVE_DEST, exist_ok=True)

shutil.copy(OUTPUT_H5, DRIVE_DEST)
print(f"Dataset copied to Google Drive: {DRIVE_DEST}")

## Step 8: Quick Dataset Validation

In [ ]:
with h5py.File(OUTPUT_H5, "r") as hf:
    shot_keys = list(hf["shots"].keys())
    n_total   = len(shot_keys)

    labels     = [hf[f"shots/{k}"].attrs["label"]    for k in shot_keys]
    triggers   = [hf[f"shots/{k}"].attrs["trigger"]  for k in shot_keys]
    dtimes     = [hf[f"shots/{k}"].attrs["disruption_time"] for k in shot_keys]

    n_normal   = sum(1 for l in labels if l == 0)
    n_disrupt  = sum(1 for l in labels if l == 1)

    from collections import Counter
    trigger_counts = Counter(t for t, l in zip(triggers, labels) if l == 1)

    # Sample shot shape
    sample = hf[f"shots/{shot_keys[0]}"]
    T_e_shape = sample["signals/T_e"].shape
    n_t = len(sample["time"])

print(f"Dataset summary:")
print(f"  Total shots     : {n_total}")
print(f"  Normal  (L=0)   : {n_normal}")
print(f"  Disrupt (L=1)   : {n_disrupt}")
print(f"  Trigger types   : {dict(trigger_counts)}")
print(f"  Te shape [n_rho, n_t] in sample shot: {T_e_shape}")

import numpy as np
valid_dtimes = [d for d, l in zip(dtimes, labels) if l==1 and not np.isnan(d)]
if valid_dtimes:
    print(f"  Disruption time : {np.mean(valid_dtimes):.1f} ± {np.std(valid_dtimes):.1f} s")